<a href="https://colab.research.google.com/github/ArinzeIhematulam/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ArinzeIhematulam/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Task type: Scoring / Ranking. Lane 2 answers "which pages should be reviewed first?" — that's the ranking/scoring pattern from the framing guide ("which ones first?" → ranking/scoring, target = a priority score, metric = precision@K). Concretely, it's implemented as a classifier predicting decline, whose probability output (predict_proba) becomes the ranking score — exactly what I did in ML-01 comparing the hand rule against the tree's Precision@K. It's a classifier under the hood, but a ranking task in purpose.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, subprocess, pandas as pd

REPO_URL = "https://github.com/ArinzeIhematulam/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")

Loaded: 30000 rows, 44 columns


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target: is_declining_label (trend_direction == "down") — a defined proxy, not an observed future outcome. It's a rule computed from the current 90-day window's trend_pct bucket, not something measured after a decision point. The lane guide itself flags this as a "beginner proxy label." A stronger capstone version would use a genuine future-window label — e.g. features from the prior 90 days predicting decline over the next 30 days, built from the warehouse's daily fact table. For this notebook I'll use is_declining_label as a working proxy, but naming it as a proxy (not ground truth) matters — treating a rule-derived label as if it were observed would mean the model just learns to copy the rule.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Metric: Precision@K (K=20 and K=50). This matches the actual decision from ML-02: an editor can only review a limited number of pages per cycle, so what matters is whether the top of the ranked list is right — not overall accuracy across all 30,000 pages, most of which nobody will ever look at.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one content item (page). content_id and client_id are pseudonymized identifiers used only for grouping and client-holdout splits — never as features.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
cols = ["content_id", "client_id", "impressions_90d", "days_since_last_update",
        "content_age_days", "avg_position", "ctr", "content_type", "is_declining_label"]

df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(f"Unit of analysis check: {df['content_id'].nunique()} unique pages across {df['client_id'].nunique()} clients")
df[cols].head(5)

Unit of analysis check: 30000 unique pages across 32 clients


,content_id,client_id,impressions_90d,days_since_last_update,content_age_days,avg_position,ctr,content_type,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,20,187,10.6,0.76,keyword article,1
1,content_a1fb4e703a9e,client_4e07408562,15320,25,445,20.3,0.05,keyword article,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,20,141,36.5,0.09,keyword article,1
3,content_331d6c4de07b,client_19581e27de,11751,22,463,6.2,0.49,keyword article,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,14,263,44.0,0.13,keyword article,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A single if-statement can check one or two conditions cleanly ("stale AND visible"), but ML-01 already showed the real signal is messier: the tree split first on impressions_90d and then on content_age_days — a threshold combination I wouldn't have picked by hand. The lane guide's own starter results back this up: the fixed baseline rule scored Precision@50 = 0.240, while a random forest reached 0.740 on the same data — roughly 3x more of the top 50 flagged pages were genuinely declining. That gap is the actual case for ML here: many weak, correlated signals (impressions, position, freshness, engagement, word count) interact in ways a hand-written rule can't capture, but a model can still be checked against precision@K and read back as reason codes.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.